# FloodValue: проверка готовности к работе

Этот ноутбук проверяет среду, форматы и арифметику на небольшом синтетическом примере. Он **не обучает модель, не оценивает паводок и не выбирает оптимальный заказ**. Все ячейки выполняются на CPU. Дальнейший маршрут — в `docs/ML_GUIDE.md`.

Для полного решения понадобится отдельный конкурсный архив организатора. Коммерческий аккаунт и секретные ключи для этого введения не нужны.

## 1. Найти репозиторий

При локальном запуске откройте ноутбук из клонированного репозитория. В Colab следующая ячейка загрузит публичный репозиторий в `/content/test_EO`. Для повторяемого эксперимента замените `REPO_REF` на сохраненный commit. Команда `git` вызывается без оболочки; ячейка не удаляет существующую папку.

In [ ]:
from pathlib import Path
import csv, json, sys, subprocess

REPO_REF = "main"  # Зафиксируйте commit перед собственным экспериментом.
starts = [Path.cwd(), Path.cwd().parent, Path("/content/test_EO")]
ROOT = next((p for p in starts if (p / "tools/pp840_helper.py").is_file()), None)
if ROOT is None:
    if not Path("/content").is_dir():
        raise RuntimeError("Запустите ноутбук из репозитория или Google Colab.")
    ROOT = Path("/content/test_EO")
    if ROOT.exists():
        raise RuntimeError("Папка /content/test_EO уже есть, но ее структура отличается. Выберите другой путь.")
    subprocess.run(["git", "clone", "https://github.com/eugenetatarchenkolaw/test_EO.git", str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", REPO_REF], check=True)
sys.path.insert(0, str(ROOT))
print("Python:", sys.version.split()[0])
print("Repository:", ROOT)
if (ROOT / ".git").exists():
    head = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True)
    print("Commit:", head.stdout.strip() if head.returncode == 0 else "еще не создан")


## 2. Проверить идентификаторы и разделение

В demo три условных независимых события. Реальные данные потребуют также проверки геометрических пересечений, совмещения растров, временной воды и лицензий. Проверка ниже этого не заменяет.

In [ ]:
from tools.audit_data import audit
from tools.pp840_helper import read_csv, price_plan, number
DATA = ROOT / "data/demo"
report = audit(DATA)
print(json.dumps(report, ensure_ascii=False, indent=2))
assert report["status"] == "synthetic_demo"


## 3. Прочитать входы

Просмотрите поля до написания модели. Стоимости активов и коэффициенты уязвимости задаются независимо от будущего прогноза. Имена `DEMO_` обозначают полностью условные записи.

In [ ]:
config = json.loads((DATA / "config.json").read_text())
catalog = read_csv(DATA / "pricing_tiles.csv")
assets = read_csv(DATA / "exposure.csv")
print("Входные поля актива:", list(assets[0]))
print("Входные поля заказа:", list(catalog[0]))
print("Бюджет примера, руб.:", config["budget_rub"])
print("Ставка примера:", catalog[0]["base_rate_rub_km2"], catalog[0]["base_rate_status"])


## 4. Пересчитать переданный заказ

Выбор ниже сделан вручную только для проверки арифметики. Он не является рекомендацией. Меняйте `selected_ids` и наблюдайте цену; скидка считается заново по всей корзине. Подробности — `docs/PRICING_PP840.md`.

In [ ]:
selected_ids = ["DEMO_O1"]
priced_rows, total = price_plan(catalog, selected_ids, config)
for row in priced_rows:
    print(json.dumps(row, ensure_ascii=False, indent=2))
print("Итого, руб.:", total)
print("В пределах бюджета:", total <= number(config["budget_rub"]))
if selected_ids == ["DEMO_O1"]:
    assert str(total) == "1703.46"  # Только опубликованный контрольный пример.


## 5. Убедиться, что очевидные ошибки отклоняются

Недоступный ID и отрицательная площадь не должны превращаться в правдоподобную цену. Это проверка контракта, а не алгоритм закупки.

In [ ]:
from copy import deepcopy
for label, rows, ids in [
    ("несуществующий ID", catalog, ["NOT_IN_CATALOG"]),
    ("повтор позиции", catalog, ["DEMO_O1", "DEMO_O1"]),
]:
    try:
        price_plan(rows, ids, config)
    except ValueError as error:
        print(label, "— обнаружено:", error)
    else:
        raise AssertionError("Ошибка не обнаружена")
bad_catalog = deepcopy(catalog)
bad_catalog[0]["area_km2"] = "-1"
try:
    price_plan(bad_catalog, ["DEMO_O1"], config)
except ValueError as error:
    print("отрицательная площадь — обнаружено:", error)
else:
    raise AssertionError("Ошибка не обнаружена")


## 6. Посмотреть научный ориентир

График построен по опубликованной таблице Sen1Floods11. Это не результат выполненного здесь обучения. Его исходные числа доступны в `data/evidence`, а различия протоколов разобраны в `docs/SCIENCE.md`.

In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(ROOT / "docs/figures/sen1floods11.svg")))


## 7. Спланировать собственный эксперимент

Запишите ответы в своей версии ноутбука:

1. Что означает целевая метка, как отделена постоянная вода и где прогноз неизвестен?
2. Как вы проверите отсутствие пересечения событий, регионов и сцен?
3. Какой простой ориентир сравните с основной моделью?
4. Какие показатели обнаружат ложные тревоги и пропуски на новом событии?
5. Как проверите вероятность и полезность неопределенности?
6. Как перенесете прогноз на фиксированный портфель?
7. Что означает полезность дополнительного наблюдения и как учтете задержку?
8. Как экономист воспроизведет цену и причины выбора?

Затем реализуйте обучение, inference, экономику и выбор заказов в собственной структуре проекта. Начинайте с одного проверенного batch, сохраняйте параметры и веса. Архитектура и интерфейс свободны.

## 8. Проверить свою сдачу

После получения конкурсного архива и формирования результатов:

```bash
python -m tools.validate_submission --data data/competition --submission outputs/team
```

Пустые таблицы из `examples/empty_submission` не являются готовой сдачей. Успешный запуск валидатора не заменяет научную проверку и не гарантирует баллы. Презентация, карта и экономическая интерпретация остаются самостоятельной работой команды.
